<a href="https://colab.research.google.com/github/aryasure/BA305-UK-Mapping-Musem-Project/blob/main/Museums_Cleaned_Datset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Cole: cleaning and preprocessing

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv('museums.csv')
df

,museum_id,Name_of_museum,Address_line_1,Address_line_2,"Village,_Town_or_City",Postcode,Latitude,Longitude,Admin_area,Accreditation,...,Area_Deprivation_index_housing,Area_Deprivation_index_income,Area_Deprivation_index_services,Area_Geodemographic_group,Area_Geodemographic_group_code,Area_Geodemographic_subgroup,Area_Geodemographic_subgroup_code,Area_Geodemographic_supergroup,Area_Geodemographic_supergroup_code,Notes
0,mm.New.1,Titanic Belfast,1 Olympic Way,NaN,Belfast,BT3 9EP,54.608080,-5.909915,/Northern Ireland/Belfast (NI Loc Gov District),Unaccredited,...,4.0,5.0,5.0,Larger Towns and Cities,2ar,Larger Towns and Cities,2a1r,Business Education and Heritage Centres,2r,NaN
1,mm.aim.1230,The Woodland Heritage Museum,NaN,Brokerswood,nr Westbury,BA13 4EH,51.270903,-2.231863,/England/South West (English Region)/Wiltshire...,Unaccredited,...,6.0,8.0,4.0,Country Living,7ar,Country Living,7a1r,Town and Country Living,7r,Previously known as Phillips Countryside Museu...
2,mm.domus.WM019,Warwickshire Museum Of Rural Life,Warwick College of Agriculture,Horticulture & Equine Studies,Moreton Morrell,CV35 9BL,52.197146,-1.555528,/England/West Midlands (English Region)/Warwic...,Unaccredited,...,5.0,8.0,1.0,English and Welsh Countryside,3ar,Older Farming Communities,3a1r,Countryside Living,3r,NaN
3,mm.aim.0484,Battle Of Flowers Museum,La Robeline,Mont des Corvees,St Ouen,JE3 2ES,49.220770,-2.107120,/Channel Islands,Unaccredited,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mm.aim.0485,Jet Age Museum,57 Crown Drive,Bishops Cleeve,Cheltenham,GL52 8TA,51.940092,-2.060766,/England/South West (English Region)/Glouceste...,Accredited,...,7.0,8.0,4.0,Country Living,7ar,Country Living,7a1r,Town and Country Living,7r,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4186,mm.domus.WA018,y Gaer,Glamorgan Street,NaN,Brecon,LD3 7DW,51.946352,-3.390436,/Wales/Powys (Welsh UA),Accredited,...,2.0,7.0,8.0,English and Welsh Countryside,3ar,Sparse English and Welsh Countryside,3a2r,Countryside Living,3r,Museum was revamped and renamed in 2019 to inc...
4187,mm.domus.SE332,Jewish Museum - Finchley,The Sternberg Centre,80 East End Road,London,N3 2SY,51.595298,-0.189303,/England/London (English Region)/Barnet (Londo...,Unaccredited,...,5.0,5.0,2.0,Ethnically Diverse Metropolitan Living,4ar,Ethnically Diverse Metropolitan Living,4a1r,Ethnically Diverse Metropolitan Living,4r,NaN
4188,mm.aim.0067,Bawdsey Radar,Transmitter Block,NaN,Bawdsey,IP12 3BA,51.991166,1.403723,/England/East of England (English Region)/Suff...,Unaccredited,...,1.0,8.0,1.0,English and Welsh Countryside,3ar,Country Living,3a2r,Countryside Living,3r,The transmitter block has been open since 2005...
4189,mm.domus.SE330,Wardown Park Museum,Wardown Park,NaN,Luton,LU2 7HA,51.894626,-0.418378,/England/East of England (English Region)/Luto...,Accredited,...,3.0,6.0,2.0,Ethnically Diverse Metropolitan Living,4ar,Ethnically Diverse Metropolitan Living,4a1r,Ethnically Diverse Metropolitan Living,4r,Hybrid (Devolved 2006 owned by local authority...


In [6]:
# Missing values by column
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_pct", ascending=False)

print(missing)

# Check exact duplicate rows and duplicate IDs
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate museum IDs:", df["museum_id"].duplicated().sum())
print("Duplicate museum names:", df["Name_of_museum"].duplicated().sum())

                                        missing_count  missing_pct  \
Notes                                            2980        71.10   
Address_line_2                                   2816        67.19   
DOMUS_Subject_Matter                             2788        66.52   
DOMUS_identifier                                 2347        56.00   
Identifier_used_in_primary_data_source           2056        49.06   
Address_line_1                                    441        10.52   
Size_provenance                                   179         4.27   
Area_Geodemographic_subgroup                       49         1.17   
Area_Geodemographic_subgroup_code                  49         1.17   
Area_Geodemographic_group                          49         1.17   
Area_Geodemographic_group_code                     49         1.17   
Area_Deprivation_index_education                   49         1.17   
Area_Deprivation_index_housing                     49         1.17   
Area_Deprivation_ind

In [7]:
#Drop unnecessary columns
columns_to_drop = [
    "DOMUS_Subject_Matter",
    "DOMUS_identifier",
    "Primary_provenance_of_data",
    "Identifier_used_in_primary_data_source",
    "Notes",
    "Size_provenance",
    "Address_line_2"
]

df = df.drop(columns=columns_to_drop)

In [8]:
# Convert column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

# Remove leading/trailing spaces in text columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Standardize blank strings as missing values
df[text_cols] = df[text_cols].replace(r"^\s*$", np.nan, regex=True)

In [9]:
#Clean museum size using the mode
df["size"] = df["size"].replace("unknown", pd.NA)
most_common_size = df["size"].mode()[0]
df["size"] = df["size"].fillna(most_common_size)

In [10]:
#Clean years columns
df["operating_status"] = np.where(
    df["year_closed"] == "9999:9999",
    "open",
    "closed"
)

# Drop the original year_closed column
df = df.drop(columns=["year_closed"])

In [11]:
#Clean subject matter
df["subject_matter"] = (
    df["subject_matter"]
    .astype("string")
    .str.strip()
    .fillna("Unknown")
)

In [12]:
#Remove invalid coordinates
df = df[
    df["latitude"].between(-90, 90) &
    df["longitude"].between(-180, 180)
].copy()


In [14]:
print(df.shape)
print(df.isnull().sum())
print(df["size"].value_counts())
df

(4190, 28)
museum_id                                0
name_of_museum                           0
address_line_1                         441
village_town_or_city                     4
postcode                                 0
latitude                                 0
longitude                                0
admin_area                               0
accreditation                            0
governance                               0
size                                     0
subject_matter                           0
year_opened                              0
area_deprivation_index                  48
area_deprivation_index_crime            48
area_deprivation_index_education        48
area_deprivation_index_employment       48
area_deprivation_index_health           48
area_deprivation_index_housing          48
area_deprivation_index_income           48
area_deprivation_index_services         48
area_geodemographic_group               48
area_geodemographic_group_code          48


,museum_id,name_of_museum,address_line_1,village_town_or_city,postcode,latitude,longitude,admin_area,accreditation,governance,...,area_deprivation_index_housing,area_deprivation_index_income,area_deprivation_index_services,area_geodemographic_group,area_geodemographic_group_code,area_geodemographic_subgroup,area_geodemographic_subgroup_code,area_geodemographic_supergroup,area_geodemographic_supergroup_code,operating_status
0,mm.New.1,Titanic Belfast,1 Olympic Way,Belfast,BT3 9EP,54.608080,-5.909915,/Northern Ireland/Belfast (NI Loc Gov District),Unaccredited,Independent-Not_for_profit,...,4.0,5.0,5.0,Larger Towns and Cities,2ar,Larger Towns and Cities,2a1r,Business Education and Heritage Centres,2r,open
1,mm.aim.1230,The Woodland Heritage Museum,NaN,nr Westbury,BA13 4EH,51.270903,-2.231863,/England/South West (English Region)/Wiltshire...,Unaccredited,Independent-Unknown,...,6.0,8.0,4.0,Country Living,7ar,Country Living,7a1r,Town and Country Living,7r,closed
2,mm.domus.WM019,Warwickshire Museum Of Rural Life,Warwick College of Agriculture,Moreton Morrell,CV35 9BL,52.197146,-1.555528,/England/West Midlands (English Region)/Warwic...,Unaccredited,Government-Local_Authority,...,5.0,8.0,1.0,English and Welsh Countryside,3ar,Older Farming Communities,3a1r,Countryside Living,3r,open
3,mm.aim.0484,Battle Of Flowers Museum,La Robeline,St Ouen,JE3 2ES,49.220770,-2.107120,/Channel Islands,Unaccredited,Independent-Private,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,closed
4,mm.aim.0485,Jet Age Museum,57 Crown Drive,Cheltenham,GL52 8TA,51.940092,-2.060766,/England/South West (English Region)/Glouceste...,Accredited,Independent-Not_for_profit,...,7.0,8.0,4.0,Country Living,7ar,Country Living,7a1r,Town and Country Living,7r,open
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4186,mm.domus.WA018,y Gaer,Glamorgan Street,Brecon,LD3 7DW,51.946352,-3.390436,/Wales/Powys (Welsh UA),Accredited,Government-Local_Authority,...,2.0,7.0,8.0,English and Welsh Countryside,3ar,Sparse English and Welsh Countryside,3a2r,Countryside Living,3r,open
4187,mm.domus.SE332,Jewish Museum - Finchley,The Sternberg Centre,London,N3 2SY,51.595298,-0.189303,/England/London (English Region)/Barnet (Londo...,Unaccredited,Independent-Not_for_profit,...,5.0,5.0,2.0,Ethnically Diverse Metropolitan Living,4ar,Ethnically Diverse Metropolitan Living,4a1r,Ethnically Diverse Metropolitan Living,4r,closed
4188,mm.aim.0067,Bawdsey Radar,Transmitter Block,Bawdsey,IP12 3BA,51.991166,1.403723,/England/East of England (English Region)/Suff...,Unaccredited,Independent-Not_for_profit,...,1.0,8.0,1.0,English and Welsh Countryside,3ar,Country Living,3a2r,Countryside Living,3r,open
4189,mm.domus.SE330,Wardown Park Museum,Wardown Park,Luton,LU2 7HA,51.894626,-0.418378,/England/East of England (English Region)/Luto...,Accredited,Government-Local_Authority,...,3.0,6.0,2.0,Ethnically Diverse Metropolitan Living,4ar,Ethnically Diverse Metropolitan Living,4a1r,Ethnically Diverse Metropolitan Living,4r,open


In [ ]:
# Save a cleaned copy
df.to_csv("museums_cleaned.csv", index=False)

print("Cleaned dataset shape:", df.shape)
print("Saved museums_cleaned.csv")

Cleaned dataset shape: (4190, 29)
Saved museums_cleaned.csv
